Instalisasi Faker

In [29]:
!pip install faker

K1 Import Library dan Inisialisasi


In [30]:
import numpy as np
import pandas as pd
from faker import Faker
import random

K2 Membuat Dataset Sintetis

In [31]:
SEED = 42
np.random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transef Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range (1, N + 1):
  trx_id = f"TRX{i:05d}"
  nama_pelanggan = fake.name()
  produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max",
                                                           "Basic", ""])
  kategori = random.choice(kategori_produk)
  harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
  qty = random.randint (1, 5)

  harga_variants = [
      str(harga_dasar),
      f"Rp{harga_dasar:,}".replace(",","."),
      f"{harga_dasar}.0",
      f"{harga_dasar}",
  ]

  harga = random.choice(harga_variants)

  tgl = fake.date_between(start_date="-90d", end_date="today")
  tgl_variants= [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d%m%Y"),
                 tgl.strftime("%d-%m-%Y")]
  tanggal = random.choice(tgl_variants)

  metode = random.choice(metode_bayar)
  if random.random() < 0.3:
    metode = metode.lower()
  if random.random() <0.2:
    kategori = kategori.upper() + " "

  kota = fake.city()
  rating = random.choice([1, 2, 3, 4, 5, None, None])

  rows.append({
      "transaction_id": trx_id,
      "customer_name": nama_pelanggan,
      "product_name": produk.strip(),
      "category": kategori,
      "price": harga,
      "quantity": qty,
      "payment_method": metode,
      "transaction_date": tanggal,
      "shipping_city": kota,
      "rating": rating,
  })


df = pd.DataFrame(rows)

for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
  idx = df.sample(frac=frac, random_state=SEED).index
  df.loc[idx, col] = np.nan

dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df_mentah = df.copy()
df_mentah.to_csv("transaksi_mentah.csv", index=False)
print ("Jumlah baris:", len(df))

Jumlah baris: 515


K3 Deteksi dan Penanganan Missing Value

In [32]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              152
dtype: int64


K4 . Deteksi dan Penanganan Duplicate

In [33]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 15
transaction_id duplicate: 15
Jumlah baris setelah drop_duplicates(): 500


Koreksi Tipe Data dan Standardisasi Format


Standardisasi teks kategorikaK5

In [34]:
for col in["category", "payment_method", "shipping_city"]:
  df[col] = df[col].astype("string").str.strip().str.title()

df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

print(df["category"].unique())
print(df["payment_method"].unique())

<StringArray>
['Elektronik', 'Fashion', 'Kesehatan', 'Rumah Tangga', 'Buku', 'Olahraga']
Length: 6, dtype: string
<StringArray>
['COD', 'Transef Bank', 'Kartu Kredit', <NA>, 'E-Wallet']
Length: 5, dtype: string


Koreksi tipe data pada kolom price

In [35]:
def bersihkan_harga(x):
  if pd.isna(x):
    return np.nan
  x = str(x).strip().replace("Rp", "").replace(".","").replace(",",".")
  try:
    return float(x)
  except ValueError:
    return np.nan
df["price"] = df["price"].apply(bersihkan_harga)

 Standardisasi format tanggal ke YYYY-MM-DD

In [36]:
def parse_tanggal(x):
  for fmt in ("%Y-%m-%d", "%d%m%Y", "%d-%m-%Y"):
    try:
      return pd.to_datetime(x, format=fmt)
    except ValueError:
      continue
  return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

 Finalisasi tipe data:

In [37]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

K6. Ekspor Dataset Bersih

In [38]:
df_bersih = df.copy()
df_bersih.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 500 baris


Menyimpan data ke dalam bentuk drive

In [39]:
import os
from google.colab import drive

drive.mount('/content/drive')
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Pratikum2"
os.makedirs(DIR_SIMPAN, exist_ok=True)

path_mentah = f"{DIR_SIMPAN}/transaksi_mentah.csv"
df_mentah.to_csv(path_mentah, index=False)

path_bersih = f"{DIR_SIMPAN}/transaksi_bersih.csv"
df_bersih.to_csv(path_bersih, index=False)

print(f"File mentah berhasil disimpan di: {path_mentah}")
print(f"File bersih berhasil disimpan di: {path_bersih}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File mentah berhasil disimpan di: /content/drive/MyDrive/BigData/Pratikum2/transaksi_mentah.csv
File bersih berhasil disimpan di: /content/drive/MyDrive/BigData/Pratikum2/transaksi_bersih.csv


Studi Kasus Praktikum

1. Kedua angka yang dihasilkan oleh tim IT (515) berbeda dengan total dari tim Finance (490) terjadi di karena data yang di peroleh oleh tim IT adalah bentuk dari data mentah penjualan bulanan. Sedangkan data yang diterima tim Finance adalah data yang telah melalui pembersihan, terdapat baris duplicate atau baris yang bernilai sama sejumlah 15 baris dan baris yang tidak memiliki nilai atau missing value sebanyak 10 baris. Baris duplicate dan missing value telah dibersihkan dengan cara dihapus, sehingga yang diterima oleh tim Finance sejumlah 490. Oleh karena itu kedua tim menerima jumlah data yang berbeda.

2. Data dengan baris total 490 lebih benar dari pada data dengan baris 515, jika dikaitkan dengan konsep Veracity. Konsep Veracity menegaskan bahwa sebuah data harus bersih dan benar sehingga prinsip GIGO(Garbage in, Garbage Out) bisa terhindarkan. Data dengan jumlah baris 515 memiliki 25 baris yang bernilai "kotor" yang menentang konsep Veracity, sehingga jika dipakai output yang dihasilkan juga "kotor".

3. Mempertahankan missing value pada kolom ratting diambil untuk menjaga keaslian data dan mencegah pembiasan analisistik statistik, dan mencegah kehilangan informasi transaksi ketika harus menghapus baris yang memiliki missing value.